# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
import math, re, tarfile, zipfile
from pathlib import Path
from collections import Counter
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ARCHIVE = Path('data/txt-files.tar.zip')
N_DOCUMENTOS = 100
def tokenize(text):
    return re.findall(r"[^\W\d_]+(?:['’][^\W\d_]+)?", text.lower(), flags=re.UNICODE)
def load_corpus(path, limit=100):
    raw, tokens = {}, {}
    with zipfile.ZipFile(path) as zf:
        tar_name = next(n for n in zf.namelist() if n.lower().endswith('.tar'))
        with zf.open(tar_name) as packed, tarfile.open(fileobj=packed, mode='r|*') as tf:
            for member in tf:
                m = re.search(r"(?:^|/)pg(\d+)\.txt$", member.name, re.I)
                if not member.isfile() or not m or f'pg{m.group(1)}.txt' in raw: continue
                stream = tf.extractfile(member)
                if stream is None: continue
                doc_id = f'pg{m.group(1)}.txt'; text = stream.read().decode('utf-8', errors='replace')
                raw[doc_id] = text; tokens[doc_id] = tokenize(text)
                if len(raw) == limit: break
    if len(raw) != limit: raise ValueError(f'Se esperaban {limit} libros; se encontraron {len(raw)}')
    return raw, tokens
raw_texts, documents = load_corpus(ARCHIVE, N_DOCUMENTOS)
doc_ids = list(documents)
def extract_title(text, doc_id):
    for line in text.splitlines()[:80]:
        m = re.match(r'\s*Title:\s*(.+)', line, re.I)
        if m: return m.group(1).strip()
    return doc_id
titles = {d: extract_title(raw_texts[d], d) for d in doc_ids}
print(f'Documentos: {len(doc_ids)} | Tokens: {sum(map(len, documents.values())):,}')
print('IDs iniciales:', doc_ids[:5])

Documentos: 100 | Tokens: 5,669,041
IDs iniciales: ['pg10000.txt', 'pg10001.txt', 'pg10002.txt', 'pg10003.txt', 'pg10004.txt']


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [2]:
def tf(term, doc_id): return Counter(documents[doc_id]).get(term, 0)
def df_term(term): return sum(term in set(tokens) for tokens in documents.values())
def idf_term(term):
    df_value = df_term(term)
    if not df_value: return None
    return math.log(len(doc_ids) / df_value) + 1

# Vocabulario determinista y conteos de frecuencia absoluta
vocabulary = sorted({term for tokens in documents.values() for term in tokens})
term_to_col = {term: i for i, term in enumerate(vocabulary)}
tf_counts = [Counter(documents[d]) for d in doc_ids]
df = Counter(term for counts in tf_counts for term in counts)
N = len(doc_ids)
idf = np.array([math.log(N / df[t]) + 1 for t in vocabulary], dtype=float)

# TF-IDF explícito como matriz dispersa (CSR)
rows, cols, values = [], [], []
for row, counts in enumerate(tf_counts):
    for term, count in counts.items():
        col = term_to_col[term]; rows.append(row); cols.append(col); values.append(count * idf[col])
X_manual = csr_matrix((values, (rows, cols)), shape=(N, len(vocabulary)), dtype=float)
lengths = [len(documents[d]) for d in doc_ids]
import pandas as pd
tabla_corpus = pd.DataFrame({'Identificador': doc_ids, 'Título': [titles[d] for d in doc_ids], 'Términos únicos': [len(tf_counts[i]) for i in range(N)]})
display(tabla_corpus)
print(f'Número de documentos: {N}; vocabulario: {len(vocabulary):,}')
print(f'Longitud de documentos (tokens) — mínima: {min(lengths):,}, media: {np.mean(lengths):,.1f}, máxima: {max(lengths):,}')
print('Ejemplo de funciones: TF(love, primer libro)=', tf('love', doc_ids[0]), '; DF(love)=', df_term('love'), '; IDF(love)=', idf_term('love'))

,Identificador,Título,Términos únicos
0,pg10000.txt,The Magna Carta,1802
1,pg10001.txt,Apocolocyntosis,2088
2,pg10002.txt,The House on the Borderland,5333
3,pg10003.txt,"My First Years as a Frenchwoman, 1876-1879",6322
4,pg10004.txt,The Warriors,6872
...,...,...,...
95,pg10087.txt,Old Lady Number 31,5396
96,pg10088.txt,"Thaumaturgia; Or, Elucidations of the Marvellous",11185
97,pg10089.txt,Elves and Heroes,3685
98,pg1008.txt,"Divine Comedy, Cary's Translation, Complete",15607


Número de documentos: 100; vocabulario: 113,070
Longitud de documentos (tokens) — mínima: 5,251, media: 56,690.4, máxima: 180,935
Ejemplo de funciones: TF(love, primer libro)= 0 ; DF(love)= 85 ; IDF(love)= 1.162518929497775


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [3]:
# A Vectorizer llamado con tokenizer recibe el mismo tokenizador; las entradas son los tokens ya procesados.
processed_texts = [' '.join(documents[d]) for d in doc_ids]
vectorizer = TfidfVectorizer(tokenizer=str.split, preprocessor=None, token_pattern=None, lowercase=False,
                            vocabulary=term_to_col, use_idf=True, smooth_idf=False,
                            sublinear_tf=False, norm=None, dtype=np.float64)
X_sklearn = vectorizer.fit_transform(processed_texts)
max_diff = float(np.max(np.abs((X_manual - X_sklearn).data))) if (X_manual - X_sklearn).nnz else 0.0
print(f'Diferencia absoluta máxima TF-IDF explícito vs. scikit-learn: {max_diff:.3e}')
print('Columnas idénticas:', np.array_equal(vectorizer.get_feature_names_out(), np.array(vocabulary)))

Diferencia absoluta máxima TF-IDF explícito vs. scikit-learn: 0.000e+00
Columnas idénticas: True


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [4]:
consultas = {
 'Q1 — Aventuras en el mar': {'necesidad': 'Encontrar relatos de navegación, barcos y vida en el océano.', 'texto': 'sea ship sailor ocean voyage whale'},
 'Q2 — Amor y relaciones': {'necesidad': 'Encontrar novelas sobre amor, matrimonio y relaciones familiares.', 'texto': 'love marriage family heart'},
 'Q3 — Ciencia y astronomía': {'necesidad': 'Encontrar textos que expliquen estrellas, planetas y observación del cielo.', 'texto': 'star planet moon sun astronomy'},
 'Q4 — Misterio y crimen': {'necesidad': 'Encontrar historias sobre detectives, asesinatos e investigación.', 'texto': 'murder detective crime mystery investigation'}
}
query_texts = [q['texto'] for q in consultas.values()]
Q = vectorizer.transform(query_texts)  # vocabulario e IDF ajustados únicamente con documentos

def cosine_manual(q, d):
    q = np.asarray(q).ravel(); d = np.asarray(d).ravel()
    denom = np.linalg.norm(q) * np.linalg.norm(d)
    return float(np.dot(q, d) / denom) if denom else 0.0

# Detalle reproducible para la primera consulta y su mejor documento
scores_manual = np.array([[cosine_manual(Q.getrow(i).toarray(), X_manual.getrow(j).toarray()) for j in range(N)] for i in range(len(consultas))])
scores_lib = cosine_similarity(Q, X_sklearn)
print('Máxima diferencia entre similitudes manuales y sklearn:', np.max(np.abs(scores_manual - scores_lib)))
best_q, best_d = 0, int(np.argmax(scores_manual[0]))
qv, dv = Q.getrow(best_q).toarray().ravel(), X_manual.getrow(best_d).toarray().ravel()
shared = np.flatnonzero((qv != 0) & (dv != 0))
parts = [(vocabulary[i], qv[i], dv[i], qv[i]*dv[i]) for i in shared]
display(pd.DataFrame(parts, columns=['Término compartido','Peso consulta','Peso documento','Producto parcial']))
print(f'Producto punto={np.dot(qv,dv):.6f}; norma consulta={np.linalg.norm(qv):.6f}; norma documento={np.linalg.norm(dv):.6f}; similitud={scores_manual[best_q,best_d]:.6f}')

Máxima diferencia entre similitudes manuales y sklearn: 2.914335439641036e-16


,Término compartido,Peso consulta,Peso documento,Producto parcial
0,ocean,1.527633,1.527633,2.333662
1,sea,1.150823,6.904937,7.946360
2,ship,1.478036,1.478036,2.184590
3,voyage,1.994252,1.994252,3.977042
4,whale,3.120264,37.443162,116.832534


Producto punto=133.274188; norma consulta=5.134676; norma documento=1536.048393; similitud=0.016898


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [5]:
def text_snippet(text, query_terms, width=220):
    lower = text.lower(); positions = [lower.find(t) for t in query_terms if t and lower.find(t) >= 0]
    start = max(0, min(positions) - width//3) if positions else 0
    return ' '.join(text[start:start+width].split()) + ('…' if start+width < len(text) else '')

rankings = {}
for qi, (name, q) in enumerate(consultas.items()):
    order = np.argsort(-scores_manual[qi])[:5]
    rows = []
    terms = tokenize(q['texto'])
    for pos, j in enumerate(order, 1):
        d = doc_ids[j]
        rows.append({'Posición': pos, 'Identificador': d, 'Título': titles[d], 'Puntuación': float(scores_manual[qi,j]), 'Fragmento': text_snippet(raw_texts[d], terms)})
    rankings[name] = pd.DataFrame(rows)
    print('\n{}: {}'.format(name, q['necesidad']))
    display(rankings[name])

print('¿Rankings idénticos dentro de tolerancia 1e-12?', np.allclose(scores_manual, scores_lib, atol=1e-12))


Q1 — Aventuras en el mar: Encontrar relatos de navegación, barcos y vida en el océano.


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,pg10015.txt,"Punchinello, Volume 1, No. 19, August 6, 1870",0.016898,"ht behind them, and, by a simultaneous impulse..."
1,2,pg10040.txt,Aleutian Indian and English Dictionary,0.013270,".....Ba-lia. Bless, to cross...............Ma-..."
2,3,pg10008.txt,The Mystery,0.012278,rations by Will Crawford_ 1907 CONTENTS PART O...
3,4,pg10081.txt,"The Boy Allies at Jutland; Or, The Greatest Na...",0.009382,"By Ensign ROBERT L. DRAKE AUTHOR OF ""The Boy A..."
4,5,pg10005.txt,A Voyage to the Moon,0.008441,The Project Gutenberg eBook of A Voyage to the...



Q2 — Amor y relaciones: Encontrar novelas sobre amor, matrimonio y relaciones familiares.


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,pg10063.txt,Happiness and Marriage,0.047443,The Project Gutenberg eBook of Happiness and M...
1,2,pg10039.txt,"The Works of Aphra Behn, Volume III",0.028194,"THE FALSE COUNT THE LUCKY CHANCE; OR, AN ALDER..."
2,3,pg10006.txt,La Fiammetta,0.026223,"Y JAMES C. BROGAN 1907. INTRODUCTION Youth, be..."
3,4,pg10085.txt,Moorish Literature,0.024131,"'en if she quitted me as soon. ""Arching eyebro..."
4,5,pg10073.txt,The Century Vocabulary Builder,0.018975,S OF VERBAL FAMILIES Words Related in Blood Ex...



Q3 — Ciencia y astronomía: Encontrar textos que expliquen estrellas, planetas y observación del cielo.


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,pg10005.txt,A Voyage to the Moon,0.012166,The Project Gutenberg eBook of A Voyage to the...
1,2,pg10002.txt,The House on the Borderland,0.011043,"oduced by Suzanne Shell, Sjaani and PG Distrib..."
2,3,pg1007.txt,"Divine Comedy, Cary's Translation, Paradise",0.007825,erg.org/ebooks/1007 Credits: Judith Smith and ...
3,4,pg10018.txt,"Punchinello, Volume 1, No. 05, April 30, 1870",0.007247,"on, Steve Schulze and the Online Distributed P..."
4,5,pg1003.txt,"Divine Comedy, Longfellow's Translation, Paradise",0.007027,mats: www.gutenberg.org/ebooks/1003 Credits: D...



Q4 — Misterio y crimen: Encontrar historias sobre detectives, asesinatos e investigación.


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,pg10082.txt,The Hampstead Mystery,0.026009,The Project Gutenberg eBook of The Hampstead M...
1,2,pg10083.txt,The House of the Whispering Pines,0.010476,"ng features thus revealed, as though to tear f..."
2,3,pg10059.txt,Aunt Jane's Nieces on Vacation,0.010279,"and perhaps a better one."" ""That will be fine,..."
3,4,pg10067.txt,The Mystery of the Boule Cabinet: A Detective ...,0.009696,The Project Gutenberg eBook of The Mystery of ...
4,5,pg10029.txt,The Hunt Ball Mystery,0.009483,The Project Gutenberg eBook of The Hunt Ball M...


¿Rankings idénticos dentro de tolerancia 1e-12? True


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [6]:
# BM25: se usan los mismos documentos, tokens y las mismas cuatro consultas.
K1, B = 1.5, 0.75
avgdl = float(np.mean(lengths))
def bm25_score(query_tokens, doc_index):
    counts = tf_counts[doc_index]; dl = lengths[doc_index]; score = 0.0
    for term, qtf in Counter(query_tokens).items():
        term_df = df.get(term, 0)
        if term_df == 0: continue
        idf_bm25 = math.log(1 + (N - term_df + 0.5) / (term_df + 0.5))
        f = counts.get(term, 0)
        denom = f + K1 * (1 - B + B * dl / avgdl)
        score += idf_bm25 * (f * (K1 + 1) / denom) * qtf
    return score

bm25_rankings = {}
for name, q in consultas.items():
    query_tokens = tokenize(q['texto'])
    scores = np.array([bm25_score(query_tokens, j) for j in range(N)])
    order = np.argsort(-scores)[:5]
    bm25_rankings[name] = pd.DataFrame([{'Posición': pos, 'Identificador': doc_ids[j], 'Título': titles[doc_ids[j]], 'Puntuación BM25': float(scores[j]), 'Fragmento': text_snippet(raw_texts[doc_ids[j]], query_tokens)} for pos, j in enumerate(order, 1)])
    print('\n{} — ranking BM25'.format(name))
    display(bm25_rankings[name])

# Seis pesos TF-IDF explícitos de muestra para verificar la fórmula TF(t,d) × IDF(t).
sample_doc = doc_ids[0]
sample_terms = [t for t in vocabulary if tf(t, sample_doc) > 0][:6]
display(pd.DataFrame([{'Término': t, 'Documento': sample_doc, 'TF': tf(t, sample_doc), 'DF': df[t], 'IDF': idf[term_to_col[t]], 'TF-IDF': tf(t, sample_doc) * idf[term_to_col[t]]} for t in sample_terms]))


Q1 — Aventuras en el mar — ranking BM25


,Posición,Identificador,Título,Puntuación BM25,Fragmento
0,1,pg10079.txt,"The Atlantic Monthly, Volume 02, No. 09, July,...",9.151624,the side of the following charming inscription...
1,2,pg10015.txt,"Punchinello, Volume 1, No. 19, August 6, 1870",8.212069,"ht behind them, and, by a simultaneous impulse..."
2,3,pg10008.txt,The Mystery,8.084803,rations by Will Crawford_ 1907 CONTENTS PART O...
3,4,pg10045.txt,Dave Darrin's Second Year at Annapolis,7.527362,OND YEAR AT ANNAPOLIS *** [Illustration: Darri...
4,5,pg10076.txt,Lister's Great Adventure,7.354374,"Cartwright was not an adventurer, although his..."



Q2 — Amor y relaciones — ranking BM25


,Posición,Identificador,Título,Puntuación BM25,Fragmento
0,1,pg10063.txt,Happiness and Marriage,2.738756,The Project Gutenberg eBook of Happiness and M...
1,2,pg10073.txt,The Century Vocabulary Builder,2.626794,S OF VERBAL FAMILIES Words Related in Blood Ex...
2,3,pg10041.txt,The Rivet in Grandfather's Neck: A Comedy of L...,2.570017,"dims, nor shakes, nor mars, nor changes, Fond ..."
3,4,pg10039.txt,"The Works of Aphra Behn, Volume III",2.552326,"THE FALSE COUNT THE LUCKY CHANCE; OR, AN ALDER..."
4,5,pg10056.txt,Chinese literature,2.534363,Wives on the Joo BOOK II.--THE ODES OF SHAOU A...



Q3 — Ciencia y astronomía — ranking BM25


,Posición,Identificador,Título,Puntuación BM25,Fragmento
0,1,pg10088.txt,"Thaumaturgia; Or, Elucidations of the Marvellous",9.918888,eque nationale de France (BnF/Gallica) at http...
1,2,pg10018.txt,"Punchinello, Volume 1, No. 05, April 30, 1870",8.666006,"on, Steve Schulze and the Online Distributed P..."
2,3,pg10005.txt,A Voyage to the Moon,7.535839,The Project Gutenberg eBook of A Voyage to the...
3,4,pg10060.txt,Discourses: Biological & Geological,5.375451,", Stan Goodman, Richard Prairie and PG Distrib..."
4,5,pg1008.txt,"Divine Comedy, Cary's Translation, Complete",5.212620,erg.org/ebooks/1008 Credits: Judith Smith and ...



Q4 — Misterio y crimen — ranking BM25


,Posición,Identificador,Título,Puntuación BM25,Fragmento
0,1,pg10082.txt,The Hampstead Mystery,11.085649,The Project Gutenberg eBook of The Hampstead M...
1,2,pg10029.txt,The Hunt Ball Mystery,9.838805,The Project Gutenberg eBook of The Hunt Ball M...
2,3,pg10083.txt,The House of the Whispering Pines,9.819859,"ng features thus revealed, as though to tear f..."
3,4,pg10017.txt,"Punchinello, Volume 1, No. 23, September 3, 1870",8.480304,"SAU STREET, NEW YORK. * * * * * THE MYSTERY OF..."
4,5,pg10067.txt,The Mystery of the Boule Cabinet: A Detective ...,8.067758,The Project Gutenberg eBook of The Mystery of ...


,Término,Documento,TF,DF,IDF,TF-IDF
0,a,pg10000.txt,207,100,1.000000,207.000000
1,abandon,pg10000.txt,3,35,2.049822,6.149466
2,abbeys,pg10000.txt,6,4,4.218876,25.313255
3,abbot,pg10000.txt,1,11,3.207275,3.207275
4,abbots,pg10000.txt,6,2,4.912023,29.472138
5,abide,pg10000.txt,1,100,1.000000,1.000000
